# ➕ Day 03: Prefix Sum Pattern

---

## The "Precompute Once, Query Forever" Trick

Here's a scenario that comes up all the time: you have an array and someone keeps asking "what's the sum from index i to index j?" over and over again.

Naive approach: add up elements each time → O(n) per query.

Prefix sum: do O(n) work once upfront, then answer each query in O(1). That's the whole pattern.

**When you see these keywords, think prefix sum:**
- "Range sum", "subarray sum"
- "Sum equals K"
- "Product of array except self"
- "Running total"

---

## 🧠 How Prefix Sum Works

Build an array where `prefix[i]` = sum of all elements from index 0 to i.

```
nums   = [2, 4, 1, 3, 5]
prefix = [2, 6, 7, 10, 15]
         ↑  ↑  ↑   ↑   ↑
         2  2+4 +1  +3  +5

Now, sum from index 1 to 3?
= prefix[3] - prefix[0] = 10 - 2 = 8
= nums[1] + nums[2] + nums[3] = 4 + 1 + 3 = 8  ✅

Formula: sum(i, j) = prefix[j] - prefix[i-1]
         (or prefix[j] - prefix[i] + nums[i])
```

That's O(1) per query. Beautiful, right?

In [ ]:
# Building a prefix sum — this is the foundation
nums = [2, 4, 1, 3, 5]

# Method 1: Manual
prefix = [0] * len(nums)
prefix[0] = nums[0]
for i in range(1, len(nums)):
    prefix[i] = prefix[i-1] + nums[i]
print("Prefix:", prefix)  # [2, 6, 7, 10, 15]

# Method 2: With a leading 0 (often cleaner, avoids edge cases)
prefix_with_zero = [0]  # prefix_with_zero[i] = sum of nums[0:i]
for num in nums:
    prefix_with_zero.append(prefix_with_zero[-1] + num)
print("With leading 0:", prefix_with_zero)  # [0, 2, 6, 7, 10, 15]

# Range sum query: sum(i, j) = prefix[j+1] - prefix[i]
# Sum from index 1 to 3:
i, j = 1, 3
print(f"Sum({i},{j}) = {prefix_with_zero[j+1] - prefix_with_zero[i]}")  # 8

**Pro tip:** Using a prefix array with a leading 0 is almost always cleaner because you avoid the `i-1` edge case when `i=0`.

---

## 🎯 Problem 1: Range Sum Query (LC #303)

```
Given an array, handle multiple queries asking for sum from index i to j.

NumArray([−2, 0, 3, −5, 2, −1])
sumRange(0, 2) → 1    (−2 + 0 + 3)
sumRange(2, 5) → −1   (3 + −5 + 2 + −1)
```

In [ ]:
class NumArray:
    def __init__(self, nums):
        """Precompute prefix sums. Time: O(n)"""
        self.prefix = [0]
        for num in nums:
            self.prefix.append(self.prefix[-1] + num)
    
    def sumRange(self, left, right):
        """Answer range query. Time: O(1)"""
        return self.prefix[right + 1] - self.prefix[left]

na = NumArray([-2, 0, 3, -5, 2, -1])
print("sumRange(0,2):", na.sumRange(0, 2))  # 1
print("sumRange(2,5):", na.sumRange(2, 5))  # -1
print("sumRange(0,5):", na.sumRange(0, 5))  # -3

---

## 🎯 Problem 2: Subarray Sum Equals K (LC #560)

This is one of the **most important** prefix sum problems. It combines prefix sum with a hash map.

```
Given an array and integer k, return the total number of 
subarrays whose sum equals k.

Input: nums = [1, 1, 1], k = 2
Output: 2   ([1,1] starting at index 0 and [1,1] starting at index 1)
```

**The insight:** If `prefix[j] - prefix[i] = k`, then the subarray from i+1 to j sums to k. So we're really asking: "for each position j, how many earlier prefix sums equal `prefix[j] - k`?"

That's a perfect job for a hash map!

In [ ]:
def subarraySum(nums, k):
    """
    Subarray Sum Equals K
    Time: O(n), Space: O(n)
    
    Key idea: count how many previous prefix sums equal (current_prefix - k)
    """
    count = 0
    current_sum = 0
    prefix_counts = {0: 1}  # Empty prefix has sum 0, seen once
    
    for num in nums:
        current_sum += num
        
        # How many times have we seen (current_sum - k) as a prefix?
        # Each occurrence means there's a subarray summing to k
        if current_sum - k in prefix_counts:
            count += prefix_counts[current_sum - k]
        
        prefix_counts[current_sum] = prefix_counts.get(current_sum, 0) + 1
    
    return count

print(subarraySum([1, 1, 1], 2))          # 2
print(subarraySum([1, 2, 3], 3))           # 2 (subarray [1,2] and [3])
print(subarraySum([1, -1, 1, 1, -1], 0))  # 4

**This pattern** — prefix sum + hash map — is incredibly powerful. It shows up in many problems disguised differently. The `{0: 1}` initialization is crucial — it accounts for subarrays that start from index 0.

---

## 🎯 Problem 3: Product of Array Except Self (LC #238)

```
Return an array where output[i] = product of all elements except nums[i].
Do it WITHOUT division and in O(n).

Input: [1, 2, 3, 4]
Output: [24, 12, 8, 6]
```

**The trick:** Build a left-product prefix and a right-product suffix, then multiply them.

```
nums    = [1,  2,  3,  4]
left    = [1,  1,  2,  6]    ← product of everything to the LEFT
right   = [24, 12, 4,  1]    ← product of everything to the RIGHT
answer  = [24, 12, 8,  6]    ← left[i] × right[i]
```

In [ ]:
def productExceptSelf(nums):
    """
    Product of Array Except Self
    Time: O(n), Space: O(1) extra (output array doesn't count)
    """
    n = len(nums)
    result = [1] * n
    
    # Pass 1: Build left products into result
    left_product = 1
    for i in range(n):
        result[i] = left_product
        left_product *= nums[i]
    
    # Pass 2: Multiply by right products
    right_product = 1
    for i in range(n - 1, -1, -1):
        result[i] *= right_product
        right_product *= nums[i]
    
    return result

print(productExceptSelf([1, 2, 3, 4]))  # [24, 12, 8, 6]
print(productExceptSelf([-1, 1, 0, -3, 3]))  # [0, 0, 9, 0, 0]

This is a really elegant solution. Two passes, no division, O(1) extra space. Interviewers love this problem because it tests if you can think beyond the obvious.

---

## 🗺️ Prefix Sum Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  PREFIX SUM VARIATIONS                                                ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  1. BASIC PREFIX SUM                                                  ║
║     Precompute cumulative sums, answer range queries O(1)            ║
║     → Range Sum Query, Running Sum                                   ║
║                                                                       ║
║  2. PREFIX SUM + HASH MAP                                            ║
║     Count/find subarrays with target sum                             ║
║     → Subarray Sum Equals K, Longest Subarray with Sum K            ║
║                                                                       ║
║  3. LEFT-RIGHT PRODUCT/SUM                                           ║
║     Build prefix from left AND right, combine                        ║
║     → Product Except Self, Trapping Rain Water                       ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Variation |
|---|---------|------------|----------|
| 303 | Range Sum Query - Immutable | Easy | Basic prefix |
| 560 | Subarray Sum Equals K | Medium | Prefix + hash map |
| 238 | Product of Array Except Self | Medium | Left-right product |
| 523 | Continuous Subarray Sum | Medium | Prefix + hash map |
| 974 | Subarray Sums Divisible by K | Medium | Prefix + modulo |
| 1480 | Running Sum of 1D Array | Easy | Basic prefix |

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Prefix sum = precompute cumulative sums for O(1) range queries│
├─────────────────────────────────────────────────────────────────────┤
│  📌 Use leading 0: prefix = [0], then append running sums         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 prefix + hash map = count subarrays with sum K in O(n)        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Don't forget {0: 1} initialization for the hash map!          │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Left-right trick: two passes (forward + backward)             │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Binary Search

The O(log n) pattern that turns impossibly large searches into 30 steps.

---

### 🎉 Prefix Sum Mastered!
You now have a powerful preprocessing technique in your arsenal.